In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
raw_dir = '../data/raw'

In [3]:
movies = pd.read_csv(f'{raw_dir}/movies.csv')
genome_scores = pd.read_csv(f'{raw_dir}/genome-scores.csv')

In [4]:
dna_matrix = genome_scores.pivot(index='movieId', columns='tagId', values='relevance')

# Not all movies have genome scores. We filter our movies df to only include the ones that do.
valid_movie_ids = dna_matrix.index
content_df = movies[movies['movieId'].isin(valid_movie_ids)].copy()

In [5]:
def get_content_recommendations(movie_title, top_n=5):
    """Returns top N similar movies based purely on Tag Genome DNA."""
    
    # 1. Find the exact movieId using string containment
    idx_search = content_df[content_df['title'].str.contains(movie_title, case=False, na=False)]
    
    if idx_search.empty:
        return f"Movie '{movie_title}' not found in the Genome database."
    
    # Extract the ID of the first match
    target_movie_id = idx_search.iloc[0]['movieId']
    exact_title = idx_search.iloc[0]['title']
    print(f"Analyzing DNA and routing recommendations for: {exact_title}\n")
    
    # 2. Extract the DNA vector for the target movie
    target_dna = dna_matrix.loc[target_movie_id].values.reshape(1, -1)
    
    # 3. Calculate cosine similarity against the entire DNA matrix
    # This mathematically finds the vectors pointing in the exact same direction
    sim_scores = cosine_similarity(target_dna, dna_matrix).flatten()
    
    # 4. Sort and get top N matches (excluding the movie itself, which is always 1.0)
    # argsort() sorts ascending, so we take the end of the array and reverse it
    similar_indices = sim_scores.argsort()[-(top_n+1):-1][::-1]
    
    # 5. Map the indices back to actual movie IDs and Titles
    top_movie_ids = dna_matrix.index[similar_indices]
    
    results = content_df[content_df['movieId'].isin(top_movie_ids)].copy()
    
    # Add the similarity score for transparency
    # We create a temporary mapping dataframe to ensure scores align perfectly with the IDs
    score_mapping = pd.DataFrame({
        'movieId': top_movie_ids,
        'similarity_score': sim_scores[similar_indices]
    })
    
    results = results.merge(score_mapping, on='movieId')
    results = results.sort_values('similarity_score', ascending=False)
    
    return results[['title', 'genres', 'similarity_score']]

In [6]:
test_movie = "Matrix, The"
recommendations = get_content_recommendations(test_movie)

display(recommendations)

Analyzing DNA and routing recommendations for: Matrix, The (1999)



,title,genres,similarity_score
4,Inception (2010),Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX,0.898807
3,Equilibrium (2002),Action|Sci-Fi|Thriller,0.897470
2,"Terminator, The (1984)",Action|Sci-Fi|Thriller,0.892831
1,Terminator 2: Judgment Day (1991),Action|Sci-Fi,0.890542
0,Blade Runner (1982),Action|Sci-Fi|Thriller,0.886513
